# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
import numpy as np
import optuna

from Challenge.paths import load_cv_folds
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [4]:
# Load datasets
folds = load_cv_folds(k=5)

# **Hyperparameter search**

In [5]:
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_SVDpp_Cython

optimizer = ModelOptimizer("SVDpp")

STUDY_NAME = MatrixFactorization_SVDpp_Cython.RECOMMENDER_NAME + "_v1"

In [6]:
# Only One fold, too much time to train
URM_train, URM_val = folds[0]

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    reg_strength = optuna_trial.suggest_float("reg_strength", 1e-5, 1e-2, log=True)
    params = {
        "num_factors": optuna_trial.suggest_int("num_factors", 32, 160, step=32),
        "batch_size": 512,
        "sgd_mode": "adagrad",
        "learning_rate": optuna_trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "user_reg": reg_strength,
        "item_reg": reg_strength,
        "bias_reg": reg_strength,
        "positive_reg": reg_strength,
        "negative_reg": 0.0,
        "epochs": 300,
        
        "use_bias": True
    }
    
    recommender_instance = MatrixFactorization_SVDpp_Cython(URM_train)
    recommender_instance.fit(**params)
    
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_val)

    optimizer.log_folds([score], params)

    return score

In [7]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-28 20:04:08,136] Using an existing study with name 'MatrixFactorization_SVDpp_Cython_Recommender_v1' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

SVD++: Processed 2434560 (100.0%) in 8.50 sec. MSE loss 8.48E-01. Sample per second: 286549
SVD++: Epoch 1 of 300. Elapsed time 8.18 sec
SVD++: Processed 2434560 (100.0%) in 8.55 sec. MSE loss 7.21E-01. Sample per second: 284781
SVD++: Epoch 2 of 300. Elapsed time 16.23 sec
SVD++: Processed 2434560 (100.0%) in 8.67 sec. MSE loss 6.47E-01. Sample per second: 280768
SVD++: Epoch 3 of 300. Elapsed time 24.35 sec
SVD++: Processed 2434560 (100.0%) in 9.00 sec. MSE loss 5.91E-01. Sample per second: 270521
SVD++: Epoch 4 of 300. Elapsed time 32.68 sec
SVD++: Processed 2434560 (100.0%) in 8.16 sec. MSE loss 5.46E-01. Sample per second: 298360
SVD++: Epoch 5 of 300. Elapsed time 40.84 sec
SVD++: Processed 2434560 (100.0%) in 7.89 sec. MSE loss 5.07E-01. Sample per second: 308377
SVD++: Epoch 6 of 300. Elapsed time 48.58 sec
SVD++: Processed 2434560 (100.0%) in 9.07 sec. MSE loss 4.74E-01. Sample per second: 268473
SVD++: Epoch 7 of 300. Elapsed time 56.75 sec
SVD++: Processed 2434560 (100.0%) i

/home/luigi/RecSys/Recommenders/MatrixFactorization/Cython/MatrixFactorization_Cython.py:203: SyntaxWarning: invalid escape sequence '\o'
  \operatornamewithlimits{argmin} \limits_{U,V}\frac{1}{2}||R - UV^T||^2_2 + \frac{\lambda}{2}(||U||^2_F + ||V||^2_F)
/home/luigi/RecSys/Recommenders/MatrixFactorization/Cython/MatrixFactorization_Cython.py:233: SyntaxWarning: invalid escape sequence '\o'
  \operatornamewithlimits{argmin}\limits_{x*,y*}\frac{1}{2}\sum_{i,j \in R}(r_{ij} - x_j^T \sum_{l \in R(i)} r_{il}y_l)^2 + \frac{\lambda}{2}(\sum_{i}{||x_i||^2} + \sum_{j}{||y_j||^2})


KeyboardInterrupt: 

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE